# 第 2 周 · 第 2 天作业 —— 企业宣传册（抓站 → Ollama 生成 → 译成西班牙语）

## 练习目标

用户在 Gradio 里输入公司名与官网 URL，流程是：

1. 抓取落地页与相关链接内容（`modified_scrapper`）
2. 用本地 **Ollama**（OpenAI 兼容端点）生成英文 Markdown 宣传册
3. 再流式翻译成**西班牙语**并在笔记本 / UI 中展示

## 和本课概念的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| 本地 Ollama + OpenAI SDK | `base_url=http://localhost:11434/v1` |
| JSON 结构化输出 | `response_format={"type": "json_object"}` 选相关链接 |
| Gradio Interface | 公司名 / URL / 模型下拉 → 生成宣传册 |
| 流式输出 | 翻译阶段 `stream=True` + `update_display` |

## 怎么跑

1. 本机启动 Ollama，并拉取下拉框里的模型（如 `llama3.2`）
2. 确保同目录有可用的 `modified_scrapper.py`
3. 运行各单元格后，最后一格 `view.launch()` 打开界面


In [ ]:
# ========== 导入：抓站、Ollama 客户端、Gradio UI ==========

# 标准库 os：本练习导入后按原样保留（环境相关可扩展）
import os
# json：把模型返回的链接 JSON 解析成 Python 字典
import json
# gradio：搭建「公司名 + URL + 模型」输入界面
import gradio as gr
# IPython 显示工具：流式翻译时用 display / update_display 刷新 Markdown
from IPython.display import Markdown, display, update_display
# 自定义抓取模块：取页面链接列表、取页面正文（模块名拼写 scrapper 保持原样）
from modified_scrapper import fetch_website_links, fetch_website_contents
# OpenAI 客户端类：这里用来打本地 Ollama 的兼容端点
from openai import OpenAI


In [ ]:
# ========== 常量：Ollama 地址、默认模型、客户端 ==========

# Ollama 的 OpenAI 兼容 API 根地址（本机默认 11434）
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# 默认聊天模型名（需本机已 pull）
MODEL = "llama3.2"
# 创建指向 Ollama 的客户端；本地常用占位 api_key='ollama'
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== System Prompt：让模型从链接列表里挑「宣传册相关页」 ==========

# 影响行为的英文 prompt 必须原样保留；要求 JSON 输出格式也不可改
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""


In [ ]:
# ========== 构造 user prompt：网站 URL + 抓到的链接列表 ==========

def get_links_user_prompt(url):
    # 英文指令原文保留：请模型选出与宣传册相关的完整 https 链接
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    # 调用抓取函数拿到该站所有链接
    links = fetch_website_links(url)
    # 把链接逐行拼进 prompt
    user_prompt += "\n".join(links)
    return user_prompt


In [ ]:
# ========== 调用 Ollama：以 JSON 模式选出相关链接 ==========

def select_relevant_links(url):
    # 进度日志：正在为哪个 URL、用哪个 MODEL 选链接
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    # chat.completions + response_format=json_object，强制结构化输出
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    # 取出模型返回的 JSON 文本
    result = response.choices[0].message.content
    # 解析为字典，形如 {"links": [...]}
    links = json.loads(result)
    # 打印找到的相关链接数量
    print(f"Found {len(links['links'])} relevant links")
    return links


In [ ]:
# ========== 聚合内容：落地页正文 + 各相关链接正文 ==========

def fetch_page_and_all_relevant_links(url):
    # 先抓首页 / 落地页正文
    contents = fetch_website_contents(url)
    # 再让模型挑相关链接
    relevant_links = select_relevant_links(url)
    # 用 Markdown 标题拼出「落地页 + 相关页」大文本，供后续写宣传册
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        # 每个相关链接加三级标题（type 如 about page）
        result += f"\n\n### Link: {link['type']}\n"
        # 抓取该链接正文并追加
        result += fetch_website_contents(link["url"])
    return result


In [ ]:
# ========== System Prompt：根据多页内容写短宣传册 ==========

# 英文 system 指令保留：受众、Markdown、文化/客户/招聘等要素
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [ ]:
# ========== 构造宣传册 user prompt，并截断到 5000 字符 ==========

def get_brochure_user_prompt(company_name, url):
    # 英文 user 指令：公司名 + 页面内容 → 短 Markdown 宣传册
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    # 拼上抓取聚合后的页面正文
    user_prompt += fetch_page_and_all_relevant_links(url)
    # 截断：避免上下文过长撑爆小模型窗口（阈值 5_000 保持原样）
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt


In [ ]:
# ========== 生成英文宣传册，再流式翻译成西班牙语 ==========

def stream_brochure(company_name, url, ollama_model):
    # 第一步：非流式生成英文宣传册（模型名来自 UI 下拉）
    response = ollama.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ]
    )
    # 取出英文宣传册正文
    result = response.choices[0].message.content

    # 第二步：流式翻译；user 指令英文原文保留
    stream = ollama.chat.completions.create(
        model=ollama_model,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": "Translate the following marketing brochure to Spanish: " + result}
          ],
        stream=True
    )
    # 累加流式 delta，并用 display_id 原地刷新 Markdown
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        # or ''：防止某些 chunk 的 delta.content 为 None
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)


In [ ]:
# ========== Gradio Interface：输入公司/URL/模型并启动 ==========

# 公司名输入框（label/info 英文 UI 文案按原样保留）
company_input = gr.Textbox(label="Your company:", info="Enter the company name")
# 官网 URL 输入框
url_input = gr.Textbox(label="Your URL:", info="Enter the company URL")
# 模型下拉：可选 llama3.1:8b 或 llama3.2，默认前者
model_selector = gr.Dropdown(["llama3.1:8b", "llama3.2"], label="Select model", value="llama3.1:8b")
# Markdown 输出区（展示宣传册/译文；与函数返回值的绑定保持原样）
message_output = gr.Markdown(label="Response:")

# Interface：把三个输入接到 stream_brochure
view = gr.Interface(
    fn=stream_brochure,
    title="Generate Company Brochure and Translate to Spanish", 
    inputs=[company_input, url_input, model_selector], 
    outputs=[message_output], 
    examples=[
            ["Nebula", "https://www.nebula.io/", "llama3.2"]
        ], 
    flagging_mode="never"
    )
# 启动 Gradio 服务
view.launch()
